<h1> Genetic Programming (GP) </h1>

Philosophically, this algorithm differs fundamentally from GA and DE.

So far:

GA → Finding the best solution</br>
DE → Finding the best parameter values

But GP:

It attempts to generate a program, formula, or computational structure itself and Our answer is no longer a number or a vector, but rather a program tree, it meansWe find the program or function itself that solves the problem.

### Difference between GA and GP

|                | GA                 | GP                |
| -------------- | ------------------ | ----------------- |
| Goal           | Finding the Solution | Finding the Program |
| Representation | Chromosome         | Tree              |
| Gene           | Value              | Function/Operator |
| Crossover      | Gene segment exchange | Swapping Subtrees   |
| Mutation       | Changing Gene         | Changing Node/Branch |


As previously mentioned, the main difference between GP and GA/DE lies in the representation.

In GA:</br>
Solution → Chromosome → Array

In DE:</br>
Solution → Vector → [x1, x2, ...]

However, in GP:</br>
Solution → Program → Tree

Thus, our first step is to construct an expression tree.

In [14]:
from src.GP import (
    Node,
    generate_random_tree,
    fitness_function,
    tournament_selection,
    subtree_crossover
)
from problems.symbolic_regression import create_dataset

<h3> Step 1: Tree representation & Evaluation </h3>

In [15]:
tree = Node(
    "*",
    Node(
        "+",
        Node("x"),
        Node(1)
    ),
    Node("x")
)

print(tree.to_string())

((x + 1) * x)


In [16]:
tree.evaluate(3)

12

<h3> Step 2: Random Tree Generator </h3>

So far, we have managed to construct a tree manually.

However, a true GP system must be able to:

generate a large number of random programs,</br>
place them into a population,</br>
and then initiate the evolution process.

Therefore, we need a function that generates a random expression tree.

In [17]:
tree = generate_random_tree(
    max_depth=3
)

print(tree.to_string())

((1 - (2 / x)) * x)


In [18]:
for i in range(5):
    tree = generate_random_tree(
        max_depth=3
    )
    print(
        tree.to_string()
    )

(((1 / 1) * (2 * 2)) * ((x * x) / (3 / 3)))
(((1 + 3) / 3) - ((x * 3) / 3))
x
(((1 * 3) + 2) / ((3 + 3) / 3))
3


<h3> Step 3: Symbolic Regression Function </h3>

In [19]:
x_data, y_data = create_dataset(start=-5, end=5, samples=50)

print(x_data[:5])
print(y_data[:5])

[-5.         -4.79591837 -4.59183673 -4.3877551  -4.18367347]
[16.         14.40899625 12.90129113 11.47688463 10.13577676]


<h3> Step 4: Fitness Function </h3>

For the entire dataset:

We use:

Mean Squared Error (MSE)

$MSE= \frac1n \sum(y-\hat y)^2$

$x∗x+(2∗x+1)$

In [20]:
tree = Node(
    "+",
    Node(
        "*",
        Node("x"),
        Node("x")
    ),
    Node(
        "+",
        Node(
            "*",
            Node(2),
            Node("x")
        ),
        Node(1)
    )
)

In [21]:
fitness_function(
    tree,
    x_data,
    y_data,
)

0.0

<h3> Step 5: GP Selection </h3>

In GP, ​​just as in GA, the selection process is responsible for:

Selecting better programs with higher probability.</br>
Giving weaker programs a lower chance of selection.

In [22]:
population = []

for _ in range(5):
    tree = generate_random_tree(
        max_depth=3
    )
    population.append(tree)

In [23]:
fitness_values = []

for tree in population:
    fitness_values.append(
        fitness_function(
            tree,
            x_data,
            y_data,
        )
    )

In [24]:
for i in range(len(population)):
    print(
        population[i].to_string(),
        "->",
        fitness_values[i]
    )

(((3 / 3) / (1 - x)) + x) -> 170.72658300964565
(1 + ((x - 2) * (x + 2))) -> 50.693877551020414
((2 / (2 + 3)) + ((3 / 1) / (3 / x))) -> 154.78171204175135
(3 - ((x / 2) + (x - 3))) -> 179.8553855111391
(((x - 3) - (3 * 3)) * ((1 - 2) - (x / x))) -> 404.1359977560371


In [25]:
parent = tournament_selection(
    population,
    fitness_values,
)

print(parent.to_string())

((2 / (2 + 3)) + ((3 / 1) / (3 / x)))


<h3> Step 6: Subtree Crossover </h3>

In GP, ​​crossover is typically the primary operator.

This contrasts with GA, where:</br>
Crossover</br>
Mutation</br>
are of roughly equal importance.

In GP:</br>
Typically:</br>
Crossover ≈ 80–90%</br>
Mutation ≈ 10–20%

This is because combining good programs usually yields better results than randomly altering the structure.

In [26]:
parent1 = Node(
    "*",
    Node(
        "+",
        Node("x"),
        Node(1)
    ),
    Node("x")
)

parent2 = Node(
    "-",
    Node("x"),
    Node(5)
)

print(parent1.to_string())
print(parent2.to_string())

((x + 1) * x)
(x - 5)


In [31]:
child = subtree_crossover(
    parent1,
    parent2
)

print(child.to_string())

((x + 5) * x)
